# UltraFeedback · 1.7B parts 06–10 and 4B parts 01–20 · thinking on
Attach the supplied frozen .bin file as a Kaggle dataset input, enable Internet and GPU T4 ×2, then select Save Version → Save & Run All. This version uses Kaggle outputs for checkpoints. The first run needs only the frozen input. On subsequent runs also attach the last completed version's output ZIP or output dataset.

A rolling ZIP in /kaggle/working is rebuilt every 10 minutes and at exit. The previous ZIP stays in place until the replacement is complete. These local snapshots do not publish intermediate Kaggle versions. For earlier published outputs, lower MINUTES_PER_RUN and attach each completed version to the next run. Both GPUs draw from one queue. GPU 0 prefers 1.7B and GPU 1 prefers 4B. Completed pilot or restored parts are skipped. The job aims to exit cleanly with a downloadable checkpoint ZIP after at most 10.5 hours from setup (checked between batches) or earlier when done. Account quota and session limits still apply. Look in the saved version's Output for uf_kaggle_thinking_checkpoints.zip and assignment_status.json. A fresh version can continue the remaining parts.

In [ ]:
import time
job_started = time.monotonic()
import subprocess, sys
from pathlib import Path
worker_env = Path('/kaggle/temp/uf_worker_env_nobootstrap')
worker_env.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(worker_env)],check=True)
worker_python = str(worker_env/'bin/python')
import pip
session_pip = str(Path(pip.__file__).resolve().parent)
subprocess.run([worker_python,session_pip,'install','-q','transformers==4.57.1','accelerate==1.11.0'],check=True)
subprocess.run([worker_python,'-c','from transformers import AutoModelForCausalLM; from accelerate import Accelerator'],check=True)

In [ ]:
MODE = 'both'
FIRST_PART_BY_MODE = {'qwen3-1.7b-on': 6, 'qwen3-4b-on': 1}
MINUTES_PER_RUN = 630
ZIP_INTERVAL_MINUTES = 10

In [ ]:
import base64, hashlib, io, sys, zipfile, json, time, re, tempfile, os
from pathlib import Path
PROJECT = Path('/kaggle/temp/uf_full_code')
OUT = Path('/kaggle/working/full_500_recovery_v1')
INPUT = Path('/kaggle/input')
MODES = ('qwen3-1.7b-on','qwen3-4b-on')
matches = sorted(INPUT.rglob('uf_frozen_500_b95fe3a5d9b7.bin'))
if not matches:
    raise FileNotFoundError('Attach the private Kaggle dataset containing uf_frozen_500_b95fe3a5d9b7.bin via Add Input, then rerun setup.')
bundle = matches[0].read_bytes()
assert hashlib.sha256(bundle).hexdigest()=='b95fe3a5d9b7160ef814e281a0cb5fe5afdc8ef85605ed98c4aa372ab680e75f'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    archive.extractall(PROJECT)
sys.path.insert(0,str(PROJECT))
for module in list(sys.modules):
    if module == "src" or module.startswith("src."):
        del sys.modules[module]
from src.uf_full import FullRun,atomic
run=FullRun(PROJECT,OUT)

In [ ]:
from src.study import match_requests
from src.uf_trial import _snapshot
from src.core import digest

def restore(inputs, run):
    # Ignore completion flags supplied by inputs. Rebuild them from verified batches.
    configs = {c['config_id']:c for c in run.configs if c['config_id'] in MODES}
    known = {}
    locations = {mode:{p['prompt_id']:i for i,group in enumerate(run.parts(cfg),1) for p in group}
                 for mode,cfg in configs.items()}
    counts = {mode:set() for mode in configs}
    def accept(name, read):
        parts = Path(name).parts
        filename = parts[-1] if parts else ''
        match = re.fullmatch(r'batch-\d+-([0-9a-f]{64})\.json',filename)
        if not match: return
        raw=read()
        records=json.loads(raw)
        if not records: return
        mode=records[0]['config_id']
        if mode not in configs: return
        number=locations[mode][records[0]['prompt_id']]
        part=f'part-{number:02}'
        c=configs[mode]
        partitions=run.parts(c)
        if not 1<=number<=len(partitions): raise ValueError('Unknown partition: '+name)
        if hashlib.sha256(raw).hexdigest()!=match[1]: raise ValueError('Checkpoint checksum failed: '+name)
        dest=run.out/mode/part/filename
        prompts={p['prompt_id']:p for p in partitions[number-1]}
        for e in records:
            p=prompts[e['prompt_id']]
            expected=next(r for r in match_requests(p,c,e['candidate_a'],e['candidate_b'],e['match_name']) if r['order']==e['order'])
            if any(e.get(k)!=v for k,v in expected.items()): raise ValueError('Different experiment: '+name)
            key=(e['request_id'],e['attempt'])
            fingerprint=digest(e)
            if key in known and known[key]!=fingerprint: raise ValueError('Conflicting attempts. Attach one authoritative checkpoint history per part.')
            known[key]=fingerprint
            counts[mode].add(key)
        if not dest.exists(): atomic(dest,raw)
    for directory in (run.out,inputs):
        for path in directory.rglob('batch-*.json'):
            accept(str(path),path.read_bytes)
    for path in inputs.rglob('*.zip'):
        with zipfile.ZipFile(path) as archive:
            for name in archive.namelist():
                accept(name,lambda:archive.read(name))
    return {mode:len(keys) for mode,keys in counts.items()}

print('Mounted Kaggle inputs:', [p.name for p in INPUT.iterdir()] if INPUT.is_dir() else [])
from itertools import islice
print('Example mounted files:',[str(p.relative_to(INPUT)) for p in islice((p for p in INPUT.rglob('*') if p.is_file()),12)])
restored=restore(INPUT,run)
print('Distinct full-run attempts found:',restored)
pending=[]
for cfg in run.configs:
    mode=cfg['config_id']
    if mode not in FIRST_PART_BY_MODE: continue
    for number in range(1,len(run.parts(cfg))+1):
        if number < FIRST_PART_BY_MODE[mode]: continue
        prompts,folder,events=run.load(cfg,number)
        _,stats=_snapshot(prompts,[cfg],events)
        stat=stats[0]
        done=not(stat['semifinals_pending'] or stat['finals_pending'])
        if done: atomic(folder/'finished.json',json.dumps(stat).encode())
        elif (folder/'finished.json').exists(): (folder/'finished.json').unlink()
        print(mode,number,'Already done' if done else stat['state'], '| saved semifinal/final:',stat['semifinals_saved'],stat['finals_saved'])
        if not done: pending.append((mode,number))

In [ ]:
worker_path = PROJECT/'uf_kaggle_worker.py'
worker_path.write_text('"""One isolated GPU process per Kaggle accelerator, with exclusive part ownership."""\nimport fcntl\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef process_parts(run, jobs, deadline, judge, timeout):\n    for mode, number in jobs:\n        if time.monotonic() >= deadline:\n            break\n        folder = run.out / mode / f\'part-{number:02}\'\n        folder.mkdir(parents=True, exist_ok=True)\n        with (folder/\'.worker.lock\').open(\'a\') as lock:\n            try:\n                fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n            except BlockingIOError:\n                continue\n            if (folder/\'finished.json\').is_file():\n                continue\n            print(f"GPU {os.environ[\'CUDA_VISIBLE_DEVICES\']} starting {mode}, part {number}", flush=True)\n            try:\n                run.run(mode, number, factory=judge)\n            except timeout:\n                break\n\n\nif __name__ == \'__main__\':\n    project, output, jobs, end = sys.argv[1:]\n    sys.path.insert(0, project)\n    from src.uf_full import FullRun\n    from src.uf_trial import LocalJudge\n    import torch\n    if torch.cuda.device_count() != 1:\n        raise RuntimeError(\'Worker expected one isolated CUDA device.\')\n    print(\'Assigned accelerator:\', torch.cuda.get_device_name(0), flush=True)\n    deadline = float(end)\n\n    class TimeLimit(Exception):\n        pass\n\n    class BudgetJudge(LocalJudge):\n        def generate_batch(self, rows, config, attempt):\n            if time.monotonic() >= deadline:\n                raise TimeLimit\n            return super().generate_batch(rows, config, attempt)\n\n    process_parts(FullRun(project, output), json.loads(jobs), deadline, BudgetJudge, TimeLimit)\n')

In [ ]:
import subprocess
import threading
import torch

def save_zip():
    target=OUT.parent/'uf_kaggle_thinking_checkpoints.zip'
    scratch=target.with_suffix('.tmp')
    with zipfile.ZipFile(scratch,'w',zipfile.ZIP_DEFLATED) as archive:
        for file in OUT.rglob('*'):
            if file.is_file() and file.suffix not in ('.tmp','.lock'):
                archive.write(file,file.relative_to(OUT.parent))
    scratch.replace(target)
    print('Local checkpoint ZIP refreshed:', target, flush=True)

assert MODE in ('both',*MODES) and MINUTES_PER_RUN>0 
jobs=[(m,n) for m,n in pending if MODE in ('both',m) and not (OUT/m/f'part-{n:02}'/'finished.json').exists()]
procs=[]
readers=[]
def relay(proc, device):
    for line in proc.stdout:
        print(f'[GPU {device}] {line.rstrip()}',flush=True)

try:
    if jobs and torch.cuda.device_count()!=2:
        raise RuntimeError('Select GPU T4 x2 in Kaggle settings and restart the session.')
    visible=os.environ.get('CUDA_VISIBLE_DEVICES')
    devices=visible.split(',') if visible else ['0','1']
    deadline=job_started+60*MINUTES_PER_RUN
    for i in range(min(2,len(jobs))):
        order=sorted(jobs,key=lambda item: (item[0]!=MODES[i],item[1]))
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=devices[i].strip(),TOKENIZERS_PARALLELISM='false')
        proc=subprocess.Popen([worker_python,'-u',str(worker_path),str(PROJECT),str(OUT),json.dumps(order),str(deadline)],
                              env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        procs.append(proc)
        reader=threading.Thread(target=relay,args=(proc,i),daemon=True)
        reader.start()
        readers.append(reader)
    next_zip = time.monotonic() + 60 * ZIP_INTERVAL_MINUTES
    while any(proc.poll() is None for proc in procs):
        if time.monotonic() >= next_zip:
            save_zip()
            next_zip = time.monotonic() + 60 * ZIP_INTERVAL_MINUTES
        if any(proc.poll() not in (None,0) for proc in procs):
            raise RuntimeError('GPU worker failed. See its output above.')
        time.sleep(1)
    if any(proc.returncode for proc in procs):
        raise RuntimeError('GPU worker failed. See its output above.')
finally:
    for proc in procs:
        if proc.poll() is None: proc.terminate()
    for proc in procs:
        try: proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    for reader in readers: reader.join()
    save_zip()
remaining = [(mode,number) for mode,number in pending
             if not (OUT/mode/f'part-{number:02}'/'finished.json').is_file()]
status = {'remaining_parts': remaining, 'assignment_finished': not remaining}
(OUT.parent/'assignment_status.json').write_text(json.dumps(status,indent=2))
print(status)


Download **uf_kaggle_thinking_checkpoints.zip** from the notebook outputs. Its two model folders use the Colab checkpoint layout. For the combined report, retain this ZIP alongside your other model outputs.